# ARC Policy Runtime Kaggle Submission

This notebook is the Kaggle entrypoint for the clean `policy_runtime_submission`
runtime bundle.

It does four things:

1. Unpacks the bundled runtime and vendored ARC libraries.
2. Runs the agent against all available ARC environments.
3. Closes the scorecard and captures the final result.
4. Writes `submission.parquet` plus debugging side artifacts.

If the competition dataset exposes a sample parquet, this notebook will inspect
that schema and try to match it automatically. Otherwise it writes a minimal
4-column fallback parquet based on successful ARC submission examples.


In [ ]:
from __future__ import annotations

import base64
import io
import json
import os
import sys
import tarfile
from pathlib import Path

import pandas as pd

SOURCE_URL = "https://github.com/sparkplug604/arc-policy-runtime-submission"
RUNTIME_LABEL = "policy_runtime_submission"
MAX_STEPS_PER_ENV = 100
TEST_GAME_IDS = None  # Example: ["ls20"] for a quick dry run before a real submission.

os.environ.setdefault("OPERATION_MODE", "offline")
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

BUNDLE_ROOT = Path("/kaggle/working/arc_policy_runtime_bundle")
RUNTIME_ROOT = BUNDLE_ROOT / "policy_runtime_submission"
VENDOR_ROOT = BUNDLE_ROOT / "vendor"
BUNDLED_ENVIRONMENTS_ROOT = BUNDLE_ROOT / "environment_files"
RUNTIME_MEMORY_ROOT = RUNTIME_ROOT / "policy_memory_data"
RUNTIME_MEMORY_ROOT.mkdir(parents=True, exist_ok=True)


In [ ]:
BUNDLE_B64 = """..."""

BUNDLE_ROOT.mkdir(parents=True, exist_ok=True)
raw = base64.b64decode(BUNDLE_B64.encode("ascii"))
with tarfile.open(fileobj=io.BytesIO(raw), mode="r:gz") as archive:
    archive.extractall(BUNDLE_ROOT)

if str(RUNTIME_ROOT) not in sys.path:
    sys.path.insert(0, str(RUNTIME_ROOT))

try:
    import arc_agi  # type: ignore
    import arcengine  # type: ignore
except Exception:
    if str(VENDOR_ROOT) not in sys.path:
        sys.path.insert(0, str(VENDOR_ROOT))
    import arc_agi  # type: ignore
    import arcengine  # type: ignore

print("Bundle extracted to:", BUNDLE_ROOT)
print("Using arc_agi from:", Path(arc_agi.__file__).resolve())
print("Using arcengine from:", Path(arcengine.__file__).resolve())
print("Bundled environments root:", BUNDLED_ENVIRONMENTS_ROOT)


In [ ]:
from typing import Any

from arc_agi import Arcade, OperationMode
from arcengine import GameAction

from observation_adapter import toolkit_obs_to_arc_state
from policy_bridge import PolicyBridge
from runtime_core import ARCRuntime

ACTION_MAP = {
    "move_up": GameAction.ACTION1,
    "move_down": GameAction.ACTION2,
    "move_left": GameAction.ACTION3,
    "move_right": GameAction.ACTION4,
}


def to_toolkit_action(action_str: str) -> GameAction:
    try:
        return ACTION_MAP[action_str]
    except KeyError as exc:
        raise ValueError(f"Unsupported runtime action: {action_str}") from exc


def _get_operation_mode() -> Any:
    if hasattr(OperationMode, "OFFLINE"):
        return OperationMode.OFFLINE
    if hasattr(OperationMode, "COMPETITION"):
        return OperationMode.COMPETITION
    for mode in OperationMode:
        if str(getattr(mode, "value", mode)).lower() == "offline":
            return mode
        if str(getattr(mode, "value", mode)).lower() == "competition":
            return mode
    return getattr(OperationMode, "ONLINE", None)


def _scorecard_to_dict(scorecard: Any) -> dict[str, Any]:
    if isinstance(scorecard, dict):
        return scorecard
    if hasattr(scorecard, "model_dump"):
        return scorecard.model_dump()
    if hasattr(scorecard, "dict"):
        return scorecard.dict()
    return {"raw_scorecard_repr": repr(scorecard)}


def _environment_game_id(environment_info: Any) -> str:
    if hasattr(environment_info, "game_id"):
        return str(environment_info.game_id)
    if isinstance(environment_info, dict):
        return str(environment_info.get("game_id"))
    raise TypeError(f"Unsupported environment info type: {type(environment_info)!r}")


def _resolve_environments_dir() -> str:
    input_root = Path("/kaggle/input")
    if input_root.exists():
        for candidate in input_root.rglob("environment_files"):
            if candidate.is_dir():
                print("Using Kaggle-mounted environments:", candidate)
                return str(candidate)
        for candidate in input_root.rglob("metadata.json"):
            print("Using Kaggle input root for recursive metadata scan:", input_root)
            return str(input_root)
    print("Using bundled local environments:", BUNDLED_ENVIRONMENTS_ROOT)
    return str(BUNDLED_ENVIRONMENTS_ROOT)


def run_environment(arc: Arcade, card_id: str, game_id: str, *, max_steps: int) -> dict[str, Any]:
    brain = PolicyBridge(root=RUNTIME_MEMORY_ROOT)
    runtime = ARCRuntime(guidance_bridge=brain)
    runtime.reset(game_id)
    runtime.apply_guidance(brain.bootstrap_guidance())

    env = arc.make(game_id, scorecard_id=card_id, render_mode=None)
    if env is None:
        raise RuntimeError(f"Unable to create ARC environment for game '{game_id}'.")

    obs = env.reset()
    if obs is None:
        raise RuntimeError(f"Environment reset() returned no observation for '{game_id}'.")

    last_player_pos = None
    last_action = None
    last_mechanic_positions: list[tuple[int, int]] = []
    last_state_dict = None
    levels_completed = 0
    step_rows: list[dict[str, Any]] = []

    for step_index in range(max_steps):
        if bool(getattr(obs, "full_reset", False)) and step_index > 0:
            runtime.begin_new_attempt(game_id)
            runtime.apply_guidance(brain.bootstrap_guidance())
            last_action = None
            last_player_pos = None
            last_mechanic_positions = []
            last_state_dict = None
            step_rows.append(
                {
                    "game_id": game_id,
                    "step": step_index,
                    "event": "attempt_reset",
                }
            )
            continue

        arc_state_dict = toolkit_obs_to_arc_state(
            obs,
            level_id=game_id,
            step_id=step_index,
            previous_player_pos=last_player_pos,
            last_action=last_action,
            previous_mechanic_positions=last_mechanic_positions,
            previous_state=last_state_dict,
        )
        action_str = runtime.step(arc_state_dict)
        toolkit_action = to_toolkit_action(action_str)
        next_obs = env.step(toolkit_action)
        if next_obs is None:
            raise RuntimeError(f"Environment step() returned no observation for '{game_id}'.")

        next_state_dict = toolkit_obs_to_arc_state(
            next_obs,
            level_id=game_id,
            step_id=step_index + 1,
            previous_player_pos=arc_state_dict.get("player_pos"),
            last_action=action_str,
            previous_mechanic_positions=last_mechanic_positions,
            previous_state=arc_state_dict,
        )
        report = runtime.observe_outcome(action_str, next_state_dict)
        guidance = brain.update(report)
        runtime.apply_guidance(guidance)

        local_context = report.local_context if isinstance(report.local_context, dict) else {}
        step_rows.append(
            {
                "game_id": game_id,
                "step": step_index,
                "action": action_str,
                "toolkit_action": toolkit_action.name,
                "goal_kind": runtime.guidance.active_goal.get("kind"),
                "score_source": runtime.last_score_source,
                "levels_completed": next_state_dict.get("resources", {}).get("levels_completed"),
                "position_delta": report.delta.to_dict().get("position_delta"),
                "crossed_marker": report.delta.to_dict().get("crossed_marker"),
                "orientation_changed": report.delta.to_dict().get("orientation_changed"),
                "completion_blockers": local_context.get("completion_blockers"),
                "route_failure_reason": local_context.get("route_failure_reason"),
                "terminal": report.terminal,
            }
        )

        obs = next_obs
        next_levels_completed = int(next_state_dict.get("resources", {}).get("levels_completed", levels_completed) or 0)
        if next_levels_completed > levels_completed:
            runtime.begin_new_level(game_id)
            runtime.apply_guidance(brain.bootstrap_guidance())
            last_action = None
            last_player_pos = None
            last_mechanic_positions = []
            last_state_dict = None
            levels_completed = next_levels_completed
        else:
            levels_completed = next_levels_completed
            last_action = action_str
            last_player_pos = next_state_dict.get("player_pos")
            last_mechanic_positions = [
                tuple(point)
                for point in next_state_dict.get("resources", {}).get("mechanic_candidate_positions", [])
                if isinstance(point, list) and len(point) == 2
            ]
            last_state_dict = next_state_dict

        if report.terminal:
            break

    return {
        "game_id": game_id,
        "levels_completed": levels_completed,
        "steps_executed": len([row for row in step_rows if row.get("action")]),
        "step_rows": step_rows,
        "brain_snapshot": brain.snapshot(),
    }


def run_submission(max_steps_per_env: int = MAX_STEPS_PER_ENV) -> tuple[dict[str, Any], list[dict[str, Any]]]:
    operation_mode = _get_operation_mode()
    environments_dir = _resolve_environments_dir()
    arcade_kwargs = {
        "environments_dir": environments_dir,
    }
    if operation_mode is not None:
        arcade_kwargs["operation_mode"] = operation_mode

    arc = Arcade(**arcade_kwargs)
    tags = ["agent", RUNTIME_LABEL, "kaggle"]
    card_id = arc.open_scorecard(source_url=SOURCE_URL, tags=tags)
    print("Opened scorecard:", card_id)

    environments = arc.get_environments()
    if TEST_GAME_IDS:
        wanted = {str(game_id) for game_id in TEST_GAME_IDS}
        environments = [env for env in environments if _environment_game_id(env) in wanted]

    per_env_results: list[dict[str, Any]] = []
    try:
        for env_info in environments:
            game_id = _environment_game_id(env_info)
            print(f"Running environment: {game_id}")
            result = run_environment(arc, card_id, game_id, max_steps=max_steps_per_env)
            per_env_results.append(result)
    finally:
        final_scorecard = _scorecard_to_dict(arc.close_scorecard(card_id))

    final_scorecard["card_id"] = final_scorecard.get("card_id", card_id)
    final_scorecard["source_url"] = final_scorecard.get("source_url", SOURCE_URL)
    final_scorecard["tags"] = final_scorecard.get("tags", tags)
    return final_scorecard, per_env_results


In [ ]:
def _find_sample_submission() -> Path | None:
    input_root = Path("/kaggle/input")
    if not input_root.exists():
        return None
    for candidate in input_root.rglob("sample_submission.parquet"):
        return candidate
    for candidate in input_root.rglob("*submission*.parquet"):
        return candidate
    return None


def _build_row_from_sample(sample_df: pd.DataFrame, scorecard: dict[str, Any]) -> dict[str, Any]:
    row: dict[str, Any] = {}
    card_id = scorecard.get("card_id")
    score = scorecard.get("score")
    total_actions = scorecard.get("total_actions")
    total_levels_completed = scorecard.get("total_levels_completed")
    source_url = scorecard.get("source_url")
    tags = scorecard.get("tags")

    for column in sample_df.columns:
        lowered = column.lower()
        if "card" in lowered and "id" in lowered:
            row[column] = card_id
        elif lowered in {"score", "public_score"}:
            row[column] = score
        elif "action" in lowered:
            row[column] = total_actions
        elif "level" in lowered and "complete" in lowered:
            row[column] = total_levels_completed
        elif "source" in lowered and "url" in lowered:
            row[column] = source_url
        elif "tag" in lowered:
            row[column] = json.dumps(tags)
        elif "json" in lowered or "opaque" in lowered or "payload" in lowered:
            row[column] = json.dumps(scorecard)
        else:
            row[column] = None
    return row


def _build_minimal_submission_rows() -> list[dict[str, Any]]:
    # Multiple accepted ARC competition submission examples use this exact
    # four-column, one-row contract.
    return [
        {
            "row_id": "1_0",
            "game_id": "1",
            "end_of_game": True,
            "score": 1,
        }
    ]


def write_submission_artifacts(scorecard: dict[str, Any], per_env_results: list[dict[str, Any]]) -> None:
    sample_path = _find_sample_submission()
    if sample_path is not None:
        print("Using sample submission schema from:", sample_path)
        sample_df = pd.read_parquet(sample_path)
        row = _build_row_from_sample(sample_df, scorecard)
        submission_df = pd.DataFrame([row], columns=sample_df.columns)
    else:
        print("No sample submission parquet found. Writing minimal ARC fallback schema.")
        rows = _build_minimal_submission_rows()
        submission_df = pd.DataFrame(
            rows,
            columns=[
                "row_id",
                "game_id",
                "end_of_game",
                "score",
            ],
        )

    submission_df.to_parquet("/kaggle/working/submission.parquet", index=False)
    Path("/kaggle/working/final_scorecard.json").write_text(json.dumps(scorecard, indent=2), encoding="utf-8")
    Path("/kaggle/working/per_environment_results.json").write_text(json.dumps(per_env_results, indent=2), encoding="utf-8")

    flat_rows: list[dict[str, Any]] = []
    for env_result in per_env_results:
        for row in env_result.get("step_rows", []):
            flat_rows.append(row)
    if flat_rows:
        pd.DataFrame(flat_rows).to_parquet("/kaggle/working/step_trace.parquet", index=False)

    print("Wrote /kaggle/working/submission.parquet")


In [ ]:
final_scorecard, per_env_results = run_submission()
print(json.dumps(final_scorecard, indent=2))
write_submission_artifacts(final_scorecard, per_env_results)
